In [1]:
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
RAW, MOD, FIG = ROOT / "data" / "raw", ROOT / "models", ROOT / "outputs" / "figures"
for p in (MOD, FIG):
    p.mkdir(parents=True, exist_ok=True)

pq, csv = RAW / "sim_runs.parquet", RAW / "sim_runs.csv"
df = pd.read_parquet(pq) if pq.exists() else pd.read_csv(csv)
df = df.sort_values(["scenario", "node", "hour"]).reset_index(drop=True)

# history features per zone (past values only, so no future leakage)
g = df.groupby(["scenario", "node"])
df["depth_rise_1h"] = df["flood_depth"] - g["flood_depth"].shift(1)
df["depth_rise_3h"] = df["flood_depth"] - g["flood_depth"].shift(3)
df["requests_trend"] = df["requests"] - g["requests"].shift(3)
df["supply_burn_3h"] = g["days_of_supply"].shift(3) - df["days_of_supply"]
df["occ_trend"] = df["nearest_shelter_occ"] - g["nearest_shelter_occ"].shift(3)
df["road_loss_3h"] = g["open_road_frac"].shift(3) - df["open_road_frac"]

rain = df.drop_duplicates(["scenario", "hour"])[["scenario", "hour", "rain_mm"]].copy()
rain["rain_3h"] = rain.groupby("scenario")["rain_mm"].transform(lambda s: s.rolling(3, min_periods=1).sum())
rain["rain_6h"] = rain.groupby("scenario")["rain_mm"].transform(lambda s: s.rolling(6, min_periods=1).sum())
df = df.merge(rain[["scenario", "hour", "rain_3h", "rain_6h"]], on=["scenario", "hour"], how="left")

FEATURES = ["hour", "population", "elevation", "rain_mm", "rain_3h", "rain_6h",
            "flood_depth", "depth_rise_1h", "depth_rise_3h", "displaced", "requests", "requests_trend",
            "days_of_supply", "supply_burn_3h", "open_road_frac", "road_loss_3h",
            "shelter_dist", "nearest_shelter_occ", "occ_trend"]   # labels are NOT features
df[FEATURES] = df[FEATURES].fillna(0).astype("float32")
print("rows:", len(df), "| features:", len(FEATURES))

rows: 763200 | features: 19


In [2]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix)

healthy = df[df["failed_now"] == 0].reset_index(drop=True)      # early-warning task: not failed yet
tr_idx, te_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
                      .split(healthy, groups=healthy["scenario"]))
train, test = healthy.iloc[tr_idx], healthy.iloc[te_idx]
X_tr, y_tr = train[FEATURES], train["will_fail_within_6h"]
X_te, y_te = test[FEATURES], test["will_fail_within_6h"]
print("scenario overlap (must be 0):", len(set(train["scenario"]) & set(test["scenario"])))

def report(name, pred):
    print(f"{name:<28} precision={precision_score(y_te, pred, zero_division=0):.3f} "
          f"recall={recall_score(y_te, pred, zero_division=0):.3f} F1={f1_score(y_te, pred, zero_division=0):.3f}")

report("always 'no failure'", np.zeros(len(y_te), dtype=int))
best_t = max(np.arange(0.05, 0.8, 0.05), key=lambda t: f1_score(y_tr, (X_tr["flood_depth"] > t).astype(int)))
report(f"rule: depth > {best_t:.2f} m", (X_te["flood_depth"] > best_t).astype(int))

clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=42).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]
print("ROC-AUC:", round(roc_auc_score(y_te, proba), 3), "| PR-AUC:", round(average_precision_score(y_te, proba), 3),
      f"(random = {y_te.mean():.3f})")
for th in (0.5, 0.3):
    report(f"model @ {th}", (proba >= th).astype(int))
print(confusion_matrix(y_te, (proba >= th).astype(int)))

scenario overlap (must be 0): 0
always 'no failure'          precision=0.000 recall=0.000 F1=0.000
rule: depth > 0.35 m         precision=0.259 recall=0.458 F1=0.331
ROC-AUC: 0.994 | PR-AUC: 0.942 (random = 0.079)
model @ 0.5                  precision=0.900 recall=0.834 F1=0.865
model @ 0.3                  precision=0.838 recall=0.883 F1=0.860
[[96990  1448]
 [  996  7484]]
